# 02 — Wind power curve

**Code under test:** `fleximorpv2/models/technologies.py`

**Purpose:** Verify `_wind_power_curve` and capacity-factor logic for offshore wind.

Run cells **top to bottom**. Each markdown cell explains the **next code cell** — what it does, what to inspect in the output, and what counts as a pass.

Track overall audit progress in Obsidian (`FlexiMORP Calculation Audit.md`). These notebooks are the lab workbook, not the checklist.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("error", category=RuntimeWarning)


def _repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "fleximorpv2").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find fleximorp-project root. "
        "Open Jupyter from the repo or a notebooks/ subdirectory."
    )


_repo = _repo_root()
_audit = _repo / "notebooks" / "audit"
for path in (_repo, _audit):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from _audit_helpers import (
    REPO_ROOT,
    OUTPUT_DIR,
    SITE_OUTPUT_DIR,
    reload_fleximorp,
    assert_close,
    assert_energy_balance,
    assert_cf_bounds,
    assert_positive,
)

reload_fleximorp()
np.random.seed(42)
print(f"Repo root: {REPO_ROOT}")
print(f"Audit outputs: {OUTPUT_DIR}")
print(f"Site outputs: {SITE_OUTPUT_DIR}")

## Step 1 — Power curve shape

**Run the next cell** to print normalised power at key wind speeds (Blyth config defaults: cut-in 3 m/s, rated 12 m/s, cut-out 25 m/s).

**Pass if:**
- 0 power below cut-in and above cut-out
- Cubic rise between cut-in and rated (~0.125 at mid-point speed 7.5 m/s)
- 1.0 (rated) from rated speed to cut-out

In [ ]:
from fleximorpv2.config import load_config
from fleximorpv2.models.technologies import TechnologyModel

config = load_config("blyth")
model = TechnologyModel(config)
params = model.technology_params["wind"]

for speed in [0, 2, 3, 7.5, 12, 20, 25, 26]:
    p = model._wind_power_curve(speed, params)
    print(f"{speed:4.1f} m/s → {p:.3f} (normalised)")

## Step 2 — Constant-wind capacity factor

**Run the next cell** with synthetic `ResourceData` (constant 8 m/s, 8760 hours).

**Pass if:** CF is between 0.25 and 0.55 after wake loss (15%) and availability; `annual_energy ≈ CF × capacity × 8760` MWh/yr.

In [ ]:
from fleximorpv2.models.technologies import ResourceData

n = 8760
resource = ResourceData(
    wind_speed=np.full(n, 8.0),
    solar_irradiance=np.zeros(n),
    wave_height=np.zeros(n),
    wave_period=np.zeros(n),
    temperature=np.full(n, 10.0),
    timestamps=np.arange(n),
)
design = {"wind_capacity": 1.0, "solar_capacity": 0.0, "wave_capacity": 0.0}
perf = model.calculate_performance(design, resource)

print(perf)
assert_energy_balance(perf["annual_energy"], perf["total_capacity"], perf["capacity_factor"])